# Figure 2 – Pulse-voltage curves and dataset overview

Data are read from `data/` (written by `run_all.py`). Every figure is saved as `Figure2_<nn>_<title>.png` when it is shown.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

DATA = Path("data")
PREFIX = "Figure2"
_plt_show = plt.show
_n = [0]


def _title(fig):
    for ax in fig.axes:
        t = ax.get_title()
        if t:
            return re.sub(r"[^A-Za-z0-9]+", "_", t).strip("_")[:50]
    return "figure"


def _save_then_show(*args, **kwargs):
    for num in plt.get_fignums():
        fig = plt.figure(num)
        _n[0] += 1
        name = f"{PREFIX}_{_n[0]:02d}_{_title(fig)}.png"
        fig.savefig(name, dpi=fig.dpi, bbox_inches="tight")
        print("saved:", name)
    _plt_show(*args, **kwargs)


plt.show = _save_then_show

## Mean pulse-voltage curves at SOC 10/30/50 % for each material and comparison of the three materials at SOC 50 %

In [ ]:
samples = pd.read_csv(DATA / "samples.csv")


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

material_files = {
    "NMC": r"NMC_2.1Ah_W_5000.xlsx",
    "LMO": r"LMO_10Ah_W_5000.xlsx",
    "LFP": r"LFP_35Ah_W_5000.xlsx",
}

soc_levels = [10, 30, 50]
fixed_soc = 50

U_cols = [f"U{i}" for i in range(1, 21)]
pulse_index = np.arange(1, 21)

soc_colors = plt.cm.cividis(np.linspace(0.15, 0.85, len(soc_levels)))
material_colors = {
    "NMC": "#86BAD4",
    "LMO": "#F5B783",
    "LFP": "#E0795F",
}

fig = plt.figure(figsize=(4, 3), dpi=600)
gs = gridspec.GridSpec(2, 2)

axes = {
    "NMC": fig.add_subplot(gs[0, 0]),
    "LMO": fig.add_subplot(gs[0, 1]),
    "LFP": fig.add_subplot(gs[1, 0]),
    "MAT": fig.add_subplot(gs[1, 1]),
}

for ax in axes.values():
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis='both', which='both', length=0)

for material, path in material_files.items():
    df = samples[samples["Mat"] == material]
    ax = axes[material]

    for soc, color in zip(soc_levels, soc_colors):
        df_soc = df[df["SOC"] == soc]
        if df_soc.empty:
            continue

        U_mean = df_soc[U_cols].mean().values

        ax.plot(
            pulse_index,
            U_mean,
            lw=1.8,
            color=color
        )

    ax.set_title(material, fontsize=12, weight="bold")
    ax.set_xlabel("Pulse voltage index")
    ax.set_ylabel("Voltage (V)")
    ax.set_xticks([1, 5, 10, 15, 20])
    ax.set_ylim(2.8, 4.0)
    ax.grid(False)

ax = axes["MAT"]

for material, path in material_files.items():
    df = samples[samples["Mat"] == material]
    df_soc = df[df["SOC"] == fixed_soc]
    if df_soc.empty:
        continue

    U_mean = df_soc[U_cols].mean().values

    ax.plot(
        pulse_index,
        U_mean,
        lw=2.2,
        color=material_colors[material]
    )

ax.set_title(f"SOC = {fixed_soc}%", fontsize=12, weight="bold")
ax.set_xlabel("Pulse voltage index")
ax.set_ylabel("Voltage (V)")
ax.set_xticks([1, 5, 10, 15, 20])
ax.set_ylim(2.8, 4.0)
ax.grid(False)

labels = ["a", "b", "c", "d"]
for lab, ax in zip(labels, axes.values()):
    ax.text(
        -0.45, 1.22, lab,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top"
    )

plt.tight_layout()
plt.show()

## Dataset Sankey diagram (material → SOC range → SOH range)

Saved as the interactive file `dataset_sankey_material_SOC_SOH.html`.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"
material_files = {
    "NMC": r"NMC_2.1Ah_W_5000.xlsx",
    "LMO": r"LMO_10Ah_W_5000.xlsx",
    "LFP": r"LFP_35Ah_W_5000.xlsx",
}

soc_bins = list(range(0, 101, 10))
soc_labels = [f"{soc_bins[i]}–{soc_bins[i+1]}%" for i in range(len(soc_bins) - 1)]

soh_bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
soh_labels = ["0.5–0.6", "0.6–0.7", "0.7–0.8", "0.8–0.9", "0.9–1.0"]

material_colors = {
    "NMC": "#86BAD4",
    "LMO": "#F5B783",
    "LFP": "#E0795F",
}

soc_colors = {
    "0–10%":  "#E3F2FD",
    "10–20%": "#BBDEFB",
    "20–30%": "#90CAF9",
    "30–40%": "#64B5F6",
    "40–50%": "#42A5F5",
    "50–60%": "#2196F3",
    "60–70%": "#1E88E5",
    "70–80%": "#1976D2",
    "80–90%": "#1565C0",
    "90–100%":"#0D47A1",
}

soh_node_color = "#E6E6E6"

nodes = []
node_colors = []
links = []

def node_id(name, color):
    if name not in nodes:
        nodes.append(name)
        node_colors.append(color)
    return nodes.index(name)

for material, path in material_files.items():
    df = samples[samples["Mat"] == material].copy()

    df["SOC_bin"] = pd.cut(
        df["SOC"], bins=soc_bins, labels=soc_labels, right=False
    )
    df["SOH_bin"] = pd.cut(
        df["SOH"], bins=soh_bins, labels=soh_labels
    )

    soc_counts = df["SOC_bin"].value_counts().sort_index()

    for soc_bin, cnt in soc_counts.items():
        if cnt == 0 or pd.isna(soc_bin):
            continue

        src = node_id(material, material_colors[material])
        soc_color = soc_colors.get(str(soc_bin), "#B0BEC5")
        tgt = node_id(f"SOC {soc_bin}", soc_color)

        links.append(dict(
            source=src,
            target=tgt,
            value=int(cnt),
            color=material_colors[material]
        ))

        df_soc = df[df["SOC_bin"] == soc_bin]
        soh_counts = df_soc["SOH_bin"].value_counts().sort_index()

        for soh_bin, soh_cnt in soh_counts.items():
            if soh_cnt == 0 or pd.isna(soh_bin):
                continue

            tgt_soh = node_id(f"SOH {soh_bin}", soh_node_color)

            links.append(dict(
                source=tgt,
                target=tgt_soh,
                value=int(soh_cnt),
                color=soc_color
            ))

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=24,
        thickness=18,
        label=nodes,
        color=node_colors,
        line=dict(color="rgba(0,0,0,0)", width=0)
    ),
    link=dict(
        source=[l["source"] for l in links],
        target=[l["target"] for l in links],
        value=[l["value"] for l in links],
        color=[l["color"] for l in links]
    )
))

fig.update_layout(
    title="Dataset heterogeneity across material, SOC range, and SOH",
    font=dict(size=12),
    width=1050,
    height=540,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.write_html("dataset_sankey_material_SOC_SOH.html")
fig.show()